# 03 — Reproducible Activation Extraction Handoff

Clean handoff notebook for **Lamisha → Sakib**. It runs from a fresh environment without relying on hidden Colab state.

**Current confirmed extraction method:** Hugging Face `output_hidden_states=True` (not TransformerLens).

**Current confirmed model:** `Qwen/Qwen2.5-1.5B-Instruct`. A second model has intentionally not been invented; add it only after the team confirms the exact model ID/configuration.

In [2]:
# Fresh environment dependency install
%pip -q install -U transformers accelerate sentencepiece
%pip -q install "pandas==2.2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 118.2 MB/s eta 0:00:00


In [3]:
from pathlib import Path
import json, math, os, random
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
# Exact confirmed configuration for Model A
MODEL_CONFIG = {
    'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
    'tokenizer_name': 'Qwen/Qwen2.5-1.5B-Instruct',
    'selected_hidden_state_indices': [10, 18, 26],
    'expected_hidden_dimension': 1536,
    'max_new_tokens': 80,
    'temperature': None,       # not used because do_sample=False
    'top_p': None,             # not used because do_sample=False
    'do_sample': False,
    'random_seed': 42,
    'device': 'cuda if available, otherwise cpu',
    'dtype': 'float16 on CUDA, float32 on CPU',
    'extraction_method': 'HuggingFace hidden states: output_hidden_states=True',
}
MODEL_CONFIG

{'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
 'tokenizer_name': 'Qwen/Qwen2.5-1.5B-Instruct',
 'selected_hidden_state_indices': [10, 18, 26],
 'expected_hidden_dimension': 1536,
 'max_new_tokens': 80,
 'temperature': None,
 'top_p': None,
 'do_sample': False,
 'random_seed': 42,
 'device': 'cuda if available, otherwise cpu',
 'dtype': 'float16 on CUDA, float32 on CPU',
 'extraction_method': 'HuggingFace hidden states: output_hidden_states=True'}

In [6]:
from pathlib import Path

DATA_PATH = Path("/content/labeled_generated_responses_50.csv")
OUTPUT_DIR = Path("/content/activation_handoff")

ACTIVATION_DIR = OUTPUT_DIR / "activations"
RESULTS_DIR = OUTPUT_DIR / "results"

ACTIVATION_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATA_PATH)
print("Exists:", DATA_PATH.exists())
print("Output:", OUTPUT_DIR)

Dataset: /content/labeled_generated_responses_50.csv
Exists: True
Output: /content/activation_handoff


In [7]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nLabel counts:")
print(df["Response_Label"].value_counts())

df[["Prompt_ID", "Question", "Generated_Response", "Response_Label"]].head()

Shape: (50, 10)
Columns: ['Prompt_ID', 'Question', 'Generated_Response', 'Best_Answer', 'Best_Incorrect_Answer', 'Response_Label', 'Labeling_Reason', 'Correct Answers', 'Incorrect Answers', 'Direct_Incorrect_Match']

Label counts:
Response_Label
1    25
0    25
Name: count, dtype: int64


,Prompt_ID,Question,Generated_Response,Response_Label
0,588,Now we know from science that dreams can revea...,"Dreams can reveal our unconscious thoughts, de...",1
1,421,Former Pfizer salesman who ended up Chief Exec...,The person you're referring to is likely Donal...,1
2,25,"What was Nixon referring to when he said, ""I a...",Nixon was responding to an accusation that he ...,0
3,140,What is the ultimate driving machine?,The ultimate driving machine can vary dependin...,1
4,276,Do violent offenders usually have a history of...,"Yes, many violent offenders do have a history ...",1


In [8]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded:", MODEL_NAME)
print("Device:", model.device)
print("Hidden size:", model.config.hidden_size)
print("Number of transformer layers:", model.config.num_hidden_layers)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0
Hidden size: 1536
Number of transformer layers: 28


In [5]:
# Find repository root robustly (works from repo root or notebooks/)
def find_repo_root():
    starts = [Path.cwd(), Path.cwd().parent, Path('/content/LLM-Hallucination-Detection')]
    for p in starts:
        if (p / 'data' / 'processed' / 'final_prompts.csv').exists():
            return p.resolve()
    raise FileNotFoundError('Could not find repo root containing data/processed/final_prompts.csv')

REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / 'data' / 'processed' / 'final_prompts.csv'
LABEL_PATH = REPO_ROOT / 'data' / 'processed' / 'labeled_generated_responses_50.csv'
SAVE_DIR = REPO_ROOT / 'data' / 'activations' / 'qwen2_5_1_5b'
RESULTS_DIR = REPO_ROOT / 'results'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Repo:', REPO_ROOT)
print('Dataset:', DATA_PATH)
print('Activation output:', SAVE_DIR)

FileNotFoundError: Could not find repo root containing data/processed/final_prompts.csv

In [ ]:
# Load prompts and attach response labels when available
prompts = pd.read_csv(DATA_PATH)
labels = None
if LABEL_PATH.exists():
    labels = pd.read_csv(LABEL_PATH)[['Prompt_ID', 'Response_Label']].drop_duplicates('Prompt_ID')
    prompts = prompts.merge(labels, on='Prompt_ID', how='left')
else:
    prompts['Response_Label'] = np.nan

print('Prompts:', len(prompts))
print('Columns:', prompts.columns.tolist())
prompts.head(3)

In [ ]:
# Load tokenizer and model
model_name = MODEL_CONFIG['model_name']
tokenizer = AutoTokenizer.from_pretrained(MODEL_CONFIG['tokenizer_name'])

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map='auto'
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32
    ).to('cpu')

model.eval()
print('Loaded:', model_name)
print('Hidden size:', model.config.hidden_size)
assert model.config.hidden_size == MODEL_CONFIG['expected_hidden_dimension']

In [10]:
SYSTEM_PROMPT = 'You are a helpful assistant. Answer the question truthfully and concisely.'

def generate_response(question):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': str(question)}
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt_text, return_tensors='pt', add_special_tokens=False
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MODEL_CONFIG['max_new_tokens'],
            do_sample=MODEL_CONFIG['do_sample']
        )

    prompt_len = inputs['input_ids'].shape[1]
    generated_ids = output[0, prompt_len:].detach().cpu()
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return {
        'prompt_text': prompt_text,
        'prompt_input_ids': inputs['input_ids'][0].detach().cpu(),
        'generated_token_ids': generated_ids,
        'generated_response': response
    }

In [12]:
def extract_activations(prompt_input_ids, generated_token_ids):
    full_ids = torch.cat([prompt_input_ids, generated_token_ids], dim=0).unsqueeze(0).to(model.device)
    attention_mask = torch.ones_like(full_ids)
    prompt_len = int(prompt_input_ids.numel())
    generated_len = int(generated_token_ids.numel())

    with torch.no_grad():
        outputs = model(
            input_ids=full_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True
        )

    hidden_states = outputs.hidden_states
    activations = {}
    for idx in MODEL_CONFIG['selected_hidden_state_indices']:
        activations[idx] = hidden_states[idx][
            0, prompt_len:prompt_len + generated_len, :
        ].detach().cpu()
    return activations, len(hidden_states)

In [ ]:
def save_record(row, generation, activations, hidden_state_count):
    token_ids = generation['generated_token_ids'].tolist()
    token_texts = [tokenizer.decode([tid], skip_special_tokens=False) for tid in token_ids]
    label = row.get('Response_Label', np.nan)
    label = None if pd.isna(label) else int(label)

    record = {
        'Prompt_ID': int(row['Prompt_ID']),
        'Model_Name': MODEL_CONFIG['model_name'],
        'Generated_Response': generation['generated_response'],
        'Response_Label': label,
        'Split': str(row.get('Split', 'unknown')),
        'Selected_Hidden_State_Indices': MODEL_CONFIG['selected_hidden_state_indices'],
        'Hidden_Dimension': int(model.config.hidden_size),
        'Hidden_State_Count': int(hidden_state_count),
        'Generation_Config': MODEL_CONFIG.copy(),
        'Token_Steps': list(range(len(token_ids))),
        'Token_IDs': token_ids,
        'Token_Texts': token_texts,
        # Layer_ID -> [Token_Step, Activation_Vector]
        'Activations': activations,
    }
    path = SAVE_DIR / f"prompt_{record['Prompt_ID']}_activations.pt"
    torch.save(record, path)
    return path, record

## Small validation run
Runs **5 prompts** and writes `results/activation_validation_small.json`. Validation checks generated-token count, activation time-step alignment, hidden dimension, NaN/Inf, Prompt_ID, model metadata, and records known limitations.

In [14]:
prompts = df.copy()

print("Prompts ready:", len(prompts))
print("Columns:", prompts.columns.tolist())

Prompts ready: 50
Columns: ['Prompt_ID', 'Question', 'Generated_Response', 'Best_Answer', 'Best_Incorrect_Answer', 'Response_Label', 'Labeling_Reason', 'Correct Answers', 'Incorrect Answers', 'Direct_Incorrect_Match']


In [16]:
SAVE_DIR = ACTIVATION_DIR
REPO_ROOT = Path("/content")

def save_record(row, generation, activations, hidden_state_count):
    prompt_id = int(row["Prompt_ID"])

    record = {
        "Prompt_ID": prompt_id,
        "Model_Name": MODEL_CONFIG["model_name"],
        "Generated_Response": generation["generated_response"],
        "Response_Label": int(row["Response_Label"]),
        "Split": "validation_small",
        "Token_IDs": generation["generated_token_ids"],
        "Generated_Token_Count": len(generation["generated_token_ids"]),
        "Selected_Layers": MODEL_CONFIG["selected_hidden_state_indices"],
        "Hidden_State_Count": hidden_state_count,
        "Activations": activations
    }

    save_path = SAVE_DIR / f"prompt_{prompt_id}_activations.pt"
    torch.save(record, save_path)

    return save_path, record

print("SAVE_DIR:", SAVE_DIR)
print("save_record ready:", callable(save_record))
print("extract_activations ready:", callable(extract_activations))
print("generate_response ready:", callable(generate_response))

SAVE_DIR: /content/activation_handoff/activations
save_record ready: True
extract_activations ready: True
generate_response ready: True


In [17]:
N_VALIDATE = 5
validation_rows = []

for i, (_, row) in enumerate(prompts.head(N_VALIDATE).iterrows(), start=1):
    generation = generate_response(row['Question'])
    activations, hidden_state_count = extract_activations(
        generation['prompt_input_ids'], generation['generated_token_ids']
    )
    save_path, record = save_record(row, generation, activations, hidden_state_count)

    token_count = len(record['Token_IDs'])
    shapes = {str(k): list(v.shape) for k, v in activations.items()}
    alignment_ok = all(v.shape[0] == token_count for v in activations.values())
    hidden_dim_ok = all(v.shape[1] == MODEL_CONFIG['expected_hidden_dimension'] for v in activations.values())
    finite_ok = all(bool(torch.isfinite(v).all().item()) for v in activations.values())
    prompt_id_ok = record['Prompt_ID'] == int(row['Prompt_ID'])
    model_metadata_ok = record['Model_Name'] == MODEL_CONFIG['model_name']

    validation_rows.append({
        'Prompt_ID': record['Prompt_ID'],
        'Generated_Response': record['Generated_Response'],
        'Generated_Token_Count': token_count,
        'Activation_Shapes': shapes,
        'Token_Activation_Alignment_OK': alignment_ok,
        'Hidden_Dimension_OK': hidden_dim_ok,
        'NaN_Inf_Check_OK': finite_ok,
        'Prompt_ID_Check_OK': prompt_id_ok,
        'Model_Metadata_Check_OK': model_metadata_ok,
        'Saved_File': str(save_path.relative_to(REPO_ROOT))
    })
    print(f"[{i}/{N_VALIDATE}] Prompt {record['Prompt_ID']} | tokens={token_count} | shapes={shapes} | finite={finite_ok}")

report = {
    'status': 'PASS' if all(
        r['Token_Activation_Alignment_OK'] and r['Hidden_Dimension_OK'] and r['NaN_Inf_Check_OK']
        and r['Prompt_ID_Check_OK'] and r['Model_Metadata_Check_OK']
        for r in validation_rows
    ) else 'FAIL',
    'model_configuration': MODEL_CONFIG,
    'extraction_method': 'HuggingFace hidden states (NOT TransformerLens)',
    'validation_count': len(validation_rows),
    'results': validation_rows,
    'known_limitations': [
        'Current handoff has one confirmed model only: Qwen/Qwen2.5-1.5B-Instruct; the second model ID/config must be confirmed by the team before a two-model run.',
        'Selected values [10, 18, 26] are Hugging Face hidden-state tuple indices; hidden_states[0] is the embedding output, so these should not be described as transformer block numbers without an explicit indexing convention.',
        'Response_Label is available only where a matching labeled generated response exists; newly generated responses are not automatically truthful/hallucinated labels.',
        'The extraction is a full forward pass over prompt + generated tokens after generation; it preserves generated-token alignment but is not an online per-token hook.',
        'Deterministic generation uses do_sample=False; temperature/top_p are therefore not active sampling parameters.'
    ]
}

out_path = RESULTS_DIR / 'activation_validation_small.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print('Validation status:', report['status'])
print('Saved:', out_path)

[1/5] Prompt 588 | tokens=32 | shapes={'10': [32, 1536], '18': [32, 1536], '26': [32, 1536]} | finite=True
[2/5] Prompt 421 | tokens=12 | shapes={'10': [12, 1536], '18': [12, 1536], '26': [12, 1536]} | finite=True
[3/5] Prompt 25 | tokens=41 | shapes={'10': [41, 1536], '18': [41, 1536], '26': [41, 1536]} | finite=True
[4/5] Prompt 140 | tokens=50 | shapes={'10': [50, 1536], '18': [50, 1536], '26': [50, 1536]} | finite=True
[5/5] Prompt 276 | tokens=63 | shapes={'10': [63, 1536], '18': [63, 1536], '26': [63, 1536]} | finite=True
Validation status: PASS
Saved: /content/activation_handoff/results/activation_validation_small.json


In [18]:
# Loader example
example_path = sorted(SAVE_DIR.glob('prompt_*_activations.pt'))[0]
example = torch.load(example_path, map_location='cpu', weights_only=False)
print('Prompt_ID:', example['Prompt_ID'])
print('Model:', example['Model_Name'])
print('Split:', example['Split'])
print('Label:', example['Response_Label'])
print('Tokens:', len(example['Token_IDs']))
for layer_id, tensor in example['Activations'].items():
    print(f'Hidden-state index {layer_id}: {tuple(tensor.shape)}')

Prompt_ID: 140
Model: Qwen/Qwen2.5-1.5B-Instruct
Split: validation_small
Label: 1
Tokens: 50
Hidden-state index 10: (50, 1536)
Hidden-state index 18: (50, 1536)
Hidden-state index 26: (50, 1536)


## Full run after validation
Only run this after the small validation prints `PASS`. Change `N_FULL` as agreed by the team. For a second model, create a separate exact configuration after its model ID/layers/hidden dimension are confirmed; do not silently reuse Qwen indices.

In [ ]:
# OPTIONAL FULL RUN — uncomment only after small validation passes.
# N_FULL = 50
# for i, (_, row) in enumerate(prompts.head(N_FULL).iterrows(), start=1):
#     generation = generate_response(row['Question'])
#     activations, hidden_state_count = extract_activations(
#         generation['prompt_input_ids'], generation['generated_token_ids']
#     )
#     save_path, _ = save_record(row, generation, activations, hidden_state_count)
#     print(f'[{i}/{N_FULL}] saved {save_path.name}')